In [1]:
# import libararys 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_excel('Dataset for Data Analytics.xlsx')

# Step 1 — Handle missing values
df['CouponCode'] = df['CouponCode'].fillna('NoCoupon')

# Step 2 — Feature engineering from Date
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['HasCoupon'] = (df['CouponCode'] != 'NoCoupon').astype(int)

# Step 3 — Drop ID and leakage columns
drop_cols = ['OrderID','CustomerID','TrackingNumber',
             'ShippingAddress','TotalPrice','Date']
df = df.drop(columns=drop_cols)

# Step 4 — Encode categorical columns
for col in ['Product','PaymentMethod','ReferralSource','CouponCode']:
    df[col] = LabelEncoder().fit_transform(df[col])

# Step 5 — Define features and target
features = ['Product','UnitPrice','PaymentMethod','ReferralSource',
            'CouponCode','ItemsInCart','Month','DayOfWeek','HasCoupon','Year']
df['HighQty'] = (df['Quantity'] >= 3).astype(int)   # 1 = high, 0 = low

X = df[features]
y = df['HighQty']

# Step 6 — Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [5]:
# 1. Handle missing values
df['CouponCode'] = df['CouponCode'].fillna('NoCoupon')

# 2. Feature engineering from Date
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['HasCoupon'] = df['CouponCode'].notna().astype(int)  # before fillna

# 3. Label encode categorical columns
from sklearn.preprocessing import LabelEncoder
for col in ['Product', 'PaymentMethod', 'ReferralSource', 'CouponCode']:
    df[col] = LabelEncoder().fit_transform(df[col])

# 4. Drop ID/leakage columns
drop_cols = ['OrderID', 'CustomerID', 'TrackingNumber', 'ShippingAddress',
             'Date', 'TotalPrice', 'Quantity']  # drop target-leaking cols

# 5. Scale for Logistic Regression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

KeyError: 'Date'

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Train Logistic Regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_s, y_train)

# Make predictions
y_pred = lr_model.predict(X_test_s)

# Evaluate model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7333333333333333

Confusion Matrix:
 [[ 67  32]
 [ 32 109]]

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.68      0.68        99
           1       0.77      0.77      0.77       141

    accuracy                           0.73       240
   macro avg       0.72      0.72      0.72       240
weighted avg       0.73      0.73      0.73       240

